# LLM judge consistency — analysis

Exploratory pass over `results/runs.jsonl`. Everything the README reports comes
from `src/evaluate.py`; this notebook is for looking at the runs from angles the
summary table flattens away.

Run `python src/rank.py` and `python src/evaluate.py` before this notebook.


In [ ]:
import json, sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import metrics as M

runs = [json.loads(l) for l in (ROOT / 'results/runs.jsonl').read_text().splitlines() if l.strip()]
candidates = json.loads((ROOT / 'data/candidates.json').read_text())
ids = [c['id'] for c in candidates]
print(f'{len(runs)} runs, {len(ids)} candidates')

## Run inventory

Check the shape of what was actually collected before trusting any aggregate.


In [ ]:
df = pd.DataFrame([{
    'job': r['job_id'], 'condition': r['condition'], 'replicate': r['replicate'],
    'temperature': r['temperature'], 'variant': r['prompt_variant'],
    'latency_s': r['latency_s'],
    'prompt_tokens': r['usage']['prompt_tokens'],
    'completion_tokens': r['usage']['completion_tokens'],
    'n_ranked': len(r['ranking']) if r['ranking'] else 0,
    'issues': '; '.join(r['issues']) if r['issues'] else '',
} for r in runs])

display(df.groupby(['job', 'condition']).size().unstack(fill_value=0))
print('\nruns with issues:', (df.issues != '').sum())
df[df.issues != ''][['job', 'condition', 'replicate', 'n_ranked', 'issues']]

## Where in the list does the instability live?

Mean Kendall tau is a single number over the whole list. It hides whether two
runs disagreed about the top of the shortlist or shuffled the middle. This plots
each candidate's rank across the repeated control runs — flat lines are stable
candidates, crossing lines are where the judge is undecided.


In [ ]:
clean = [r for r in runs if r['ranking'] and not r['issues'] and len(r['ranking']) == len(ids)]
jobs = sorted({r['job_id'] for r in clean})

fig, axes = plt.subplots(1, len(jobs), figsize=(5 * len(jobs), 5), sharey=True)
axes = axes if len(jobs) > 1 else [axes]

for ax, job in zip(axes, jobs):
    base = [r for r in clean if r['job_id'] == job and r['condition'] == 'base']
    for cid in ids:
        positions = [r['ranking'].index(cid) for r in base]
        ax.plot(range(len(base)), positions, marker='o', markersize=3, alpha=0.7)
    ax.set_title(job, fontsize=10)
    ax.set_xlabel('replicate')
    ax.invert_yaxis()
axes[0].set_ylabel('rank (0 = best)')
plt.tight_layout(); plt.show()

## Does the score gap predict stability?

If the model assigns near-identical scores to two candidates, their relative
order is close to a coin flip. A useful downstream rule would be: only trust the
ordering where the score gap exceeds some threshold. This checks whether the
data supports that.


In [ ]:
base = [r for r in clean if r['condition'] == 'base']
rows = []
for job in jobs:
    runs_j = [r for r in base if r['job_id'] == job]
    if len(runs_j) < 2:
        continue
    for cid in ids:
        scores = [r['scores'].get(cid) for r in runs_j]
        scores = [s for s in scores if s is not None]
        positions = [r['ranking'].index(cid) for r in runs_j]
        rows.append({'job': job, 'id': cid,
                     'mean_score': sum(scores) / len(scores) if scores else None,
                     'rank_swing': max(positions) - min(positions)})

s = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(s.mean_score, s.rank_swing, alpha=0.6)
ax.set_xlabel('mean fit score assigned'); ax.set_ylabel('rank swing across replicates')
ax.set_title('Do mid-scoring candidates move the most?', fontsize=10)
plt.tight_layout(); plt.show()

s.sort_values('rank_swing', ascending=False).head(10)

## Shortlist churn

The operationally relevant question: if you re-ran screening tomorrow, how many
of today's top-5 would still be there?


In [ ]:
for job in jobs:
    runs_j = [r['ranking'] for r in base if r['job_id'] == job]
    if len(runs_j) < 2:
        continue
    seen = {}
    for r in runs_j:
        for cid in r[:5]:
            seen[cid] = seen.get(cid, 0) + 1
    always = [c for c, n in seen.items() if n == len(runs_j)]
    sometimes = [c for c, n in seen.items() if n < len(runs_j)]
    print(f'{job}: {len(seen)} distinct candidates appeared in a top-5 across '
          f'{len(runs_j)} runs -- {len(always)} every time, {len(sometimes)} only sometimes')
    print(f'   unstable: {sorted(sometimes)}\n')

## Notes

Record observations here as you run the experiments — especially anything that
contradicts what the summary table suggests. Conclusions that survive this
notebook belong in the README's *Findings* section; conclusions that only hold
for one job, or rest on a handful of pairs, belong in *Limitations*.
